# Notebook 1 - in-memory and single-file conversion

This notebook expands examples `01_convert_string.py` and `02_convert_file.py`. It shows how SQL text becomes SQLX actions, how metadata is moved into `config {}` blocks, how dependencies are rewritten with `${ref(...)}`, and how warnings are read from the conversion report.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from sql2sqlx import convert_file, convert_string

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()
)
EXAMPLES = ROOT / "examples"

## Convert an in-memory string

The first input contains a partitioned CTAS followed by an `INSERT`. The default policy converts the CTAS into a table action and preserves the `INSERT` as an operations action because DML lifting is opt-in.

In [ ]:
sql = """
CREATE OR REPLACE TABLE analytics.daily_orders
PARTITION BY DATE(order_ts)
OPTIONS(description = "Daily order rollup")
AS
SELECT DATE(order_ts) AS d, COUNT(*) AS n
FROM raw.orders
GROUP BY 1;

INSERT INTO analytics.order_history (d, n)
SELECT d, n FROM analytics.daily_orders;
"""

result = convert_string(sql, name="pipeline.sql")
[(file.relpath, file.action_type.value, file.action_name) for file in result.files]

## Inspect the emitted SQLX

Each output keeps the original query body except for explicit, explainable rewrites such as config extraction and dependency references.

In [ ]:
for file in result.files:
    print(f"===== {file.relpath} ({file.action_type.value}) =====")
    print(file.content)
    print()

## Plot action counts

Matplotlib is useful in migration notebooks because report fields can be turned into quick review charts for teams.

In [ ]:
counts = result.report.actions_by_type
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(counts.keys(), counts.values(), color="#4C78A8")
ax.set_title("Actions emitted from in-memory SQL")
ax.set_ylabel("count")
ax.set_xlabel("action type")
plt.show()

## Convert one file and inspect warnings

The file example demonstrates path-based conversion and warning review.

In [ ]:
file_result = convert_file(str(EXAMPLES / "sql" / "staging" / "stg_orders.sql"))
file_result.files[0].content[:800]

In [ ]:
[(warning.code, warning.line, warning.message) for warning in file_result.report.warnings]